# AI Hackathon: Weather Downscaling with Corrective Diffusion

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/maxfield-green/ai_hackathons/blob/main/notebooks/corrdiff_demo_for_hackathons.ipynb)


Train a generative weather downscaling model (**CorrDiff**) on meteorological data using NVIDIA PhysicsNeMo in Google Colab.

> **GPU is strongly recommended** (T4 or better), but this notebook includes a CPU fallback mode so users can still complete the full flow without access to a GPU.

> **This notebook is designed for Google Colab.** Code adaptations may be needed to run locally in another setting.

## Goals

By the end of this notebook, users will have gone through the following:

1. An overview of atmospheric downscaling and why it matters for high-resolution weather products.
2. An overview of CorrDiff's deterministic regression model and stochastic diffusion model. In this guide, we will only run the regression model due to time and hardware limits.
3. Train a small CorrDiff regression model on High-Resolution Rapid Refresh (HRRR) data.
4. Run inference and inspect model outputs both visually and quantitatively.
5. Try a domain-shift experiment to downscale quarter-degree GFS data over a different region of the globe.

## Background and framing

Atmospheric **downscaling** means taking coarse weather model fields (for example, global model output at ~25 to 30 km spacing) and producing finer-resolution fields (for example, ~3 km) that better represent local structure. This matters because many practical decisions depend on local weather features that coarse grids cannot resolve well, such as terrain-influenced winds, localized precipitation, and urban-scale temperature gradients.

**CorrDiff** (Corrective Diffusion Model) is a two-stage framework for km-scale atmospheric downscaling:

1. A **regression UNet** learns a supervised mapping from low-resolution inputs to a deterministic high-resolution estimate (the conditional mean).
2. A **diffusion model** then learns residual fine-scale structure around that estimate, enabling stochastic high-resolution samples and ensembles.

For workshop speed and clarity, this notebook focuses on **stage 1 only** (regression). This keeps training and inference lightweight while still demonstrating realistic weather downscaling workflows and model/data plumbing end to end.

The notebook uses a **"HRRR Mini" dataset** provided by NVIDIA in grouped NetCDF format, including:
- `input`: low-resolution predictor channels
- `output`: high-resolution target channels
- `invariant`: static geospatial fields (elevation, land-sea mask)

## How to use this notebook effectively

- Run cells top to bottom the first time. After a successful run, you can go back to change things and rerun cells as you wish.
- If a cell fails, fix the issue and re-run that cell before continuing.
- Pay attention to printed shapes, file paths, and checkpoint names; those are the fastest way to debug pipeline issues.
- Most steps are safe to re-run. Training/inference cells may overwrite outputs depending on config and runtime state.

## Runtime expectations

- **GPU runtime**: fastest and recommended.
- **CPU runtime**: supported with shorter training length so participants can still complete the exercise without access to a GPU.
- Network-dependent cells (downloads/install) may occasionally need a retry.

**Paper**: [CorrDiff — Residual Corrective Diffusion Modeling for Km-scale Atmospheric Downscaling (arXiv 2309.15214)](https://arxiv.org/abs/2309.15214)

---
## 0. Prerequisites — Check Compute Runtime

Before installing anything, verify the runtime hardware.

### Why this matters

This notebook adapts behavior based on available compute:
- **GPU mode** (preferred)
- **CPU mode** (fallback)

### What to do in Colab

1. Open **Runtime -> Change runtime type**.
2. Choose **T4 GPU** (or better) if available. T4 is typically available on free tiers, while higher-end GPUs (such as A100) may be available on paid tiers depending on availability.
3. Re-run the runtime check cell below and confirm `compute_type` is `gpu`.

If no GPU is available, continue anyway. The notebook will automatically switch to CPU-safe training settings.

### Expected output from the next cell

- Detected GPU name (if CUDA is available)
- CUDA and PyTorch versions
- `compute_type` set to either `gpu` or `cpu`

### Common issues

- If you expected GPU but see CPU, restart runtime and reselect GPU.
- Free-tier Colab instances may temporarily not have T4 capacity.
- If CUDA appears unavailable after switching runtime, rerun the cell or restart runtime.

In [ ]:
import torch

if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    compute_type = "gpu"
    print(f"GPU detected: {gpu_name}")
    print(f"CUDA version: {torch.version.cuda}")
else:
    gpu_name = "CPU"
    compute_type = "cpu"
    print("No GPU detected. Falling back to CPU mode.")
    print(
        "Tip: In Colab, try Runtime -> Change runtime type -> T4 GPU for faster training."
    )

print(f"Compute type: {compute_type}")
print(f"PyTorch version: {torch.__version__}")

---
## 1. Install Dependencies

This section installs everything needed to run CorrDiff training and inference in Colab.

### What gets installed

- PhysicsNeMo from the cloned repository
- CorrDiff example requirements from the project
- A couple of utility packages used later in the notebook

### Important notes

- You may see warning/error text from pip dependency resolution; most warnings are non-blocking in Colab.
- If installation fails hard (non-zero exit), rerun once before troubleshooting deeply.
- If a package conflict persists, restart runtime and run Section 1 again from the top.

### Quick success criteria

- The sanity-check cell imports `physicsnemo` successfully.
- A version string prints without exceptions.

In [ ]:
# Clone the PhysicsNeMo repository
!git clone --depth 1 https://github.com/NVIDIA/physicsnemo.git /content/physicsnemo 2>/dev/null || echo "Repository already cloned."

In [ ]:
# Install PhysicsNeMo
!pip install -q /content/physicsnemo

In [ ]:
# Install CorrDiff-specific dependencies
!pip install -q -r /content/physicsnemo/examples/weather/corrdiff/requirements.txt

In [ ]:
# A few libraries get missed
!pip install zarr
!pip install matplotlib
!pip install cfgrib eccodes
!pip install cartopy

In [ ]:
# Quick sanity check
import physicsnemo

print(f"PhysicsNeMo version: {physicsnemo.__version__}")

---
## 2. Download the HRRR-Mini Dataset

This section downloads a compact HRRR-Mini dataset from NVIDIA GPU Cloud (NGC) and places it in the CorrDiff data directory.

### Files used in this notebook

| File | Description |
|------|-------------|
| `hrrr_mini_train.nc` | Grouped NetCDF containing **input**, **output**, and **invariant** data |
| `stats.json` | Channel-wise normalization statistics used by the training pipeline |

### Why this matters

CorrDiff configs in this notebook assume this exact file structure and location. If files are missing or moved, training/inference commands may fail.

### Success checks

After running download cells:
- Both files should exist under the `.../data/hrrr_mini` directory
- File sizes should be non-trivial (not a few bytes — the HRRR dataset download here is about 2 GB)
- The subsequent exploration cell should list expected groups and variables

### If download fails

- Rerun the cell once (network hiccups are common in hosted notebooks).
- Confirm internet access in runtime.

In [ ]:
!mkdir /content/physicsnemo/examples/weather/corrdiff/data

In [ ]:
!mkdir /content/physicsnemo/examples/weather/corrdiff/data/hrrr_mini

In [ ]:
!wget --content-disposition \
'https://api.ngc.nvidia.com/v2/resources/org/nvidia/team/modulus/modulus_datasets-hrrr_mini/1/files?redirect=true&path=hrrr_mini/stats.json' \
--output-document '/content/physicsnemo/examples/weather/corrdiff/data/hrrr_mini/stats.json'


!wget --content-disposition \
'https://api.ngc.nvidia.com/v2/resources/org/nvidia/team/modulus/modulus_datasets-hrrr_mini/1/files?redirect=true&path=hrrr_mini/hrrr_mini_train.nc' \
--output-document '/content/physicsnemo/examples/weather/corrdiff/data/hrrr_mini/hrrr_mini_train.nc'

---
## 3. Explore the Dataset

Before training, inspect the dataset schema and a few examples so you understand what the model receives and predicts.

### What this section covers

- NetCDF group layout (`input`, `output`, `invariant`)
- Variable names and array shapes
- Time span and number of samples
- Normalization statistics for each channel
- A quick visual comparison of low-res input vs high-res target

### Why this step is important

Most downstream training errors come from data-shape or schema mismatches. A quick schema check now prevents much longer debugging later. Inspecting your data up front is one of the fastest ways to spot issues when training ML models.

### What to look for

- Input sample shape should be low resolution (8x8 in this setup).
- Output sample shape should be high resolution (64x64 in this setup).
- Invariant fields are full-domain static grids (e.g., 1059x1799) and are cropped/resampled by the training pipeline as needed.
- Variables printed here should match what training config expects.
- Upsampling factor should be sensible (typically 8x here).

### Important variable note

The HRRR-Mini stats file includes many output channels, but this mini workshop configuration focuses on a subset for speed. In the inference outputs here, the model predicts the 10m wind components (`10u`, `10v`).


In [ ]:
import xarray as xr
import json
import numpy as np

data_file = (
    "/content/physicsnemo/examples/weather/corrdiff/data/hrrr_mini/hrrr_mini_train.nc"
)
stats_file = "/content/physicsnemo/examples/weather/corrdiff/data/hrrr_mini/stats.json"

# Inspect the groups in the NetCDF file
for group in ["input", "output", "invariant"]:
    with xr.open_dataset(data_file, group=group) as ds:
        variables = list(ds.keys())
        shape = ds[variables[0]].shape
        print(f"Group '{group}': variables={variables}, sample shape={shape}")

# Top-level coordinates
with xr.open_dataset(data_file) as ds:
    n_samples = ds["time"].shape[0]
    print(f"\nTotal samples: {n_samples}")
    print(f"Time range: {ds['time'].values[0]} ... {ds['time'].values[-1]}")

In [ ]:
# Peek at the normalization stats
with open(stats_file) as f:
    stats = json.load(f)

print("Normalization statistics:")
for group in ["input", "output", "invariant"]:
    if group in stats:
        print(f"  {group}:")
        for var, s in stats[group].items():
            print(f"    {var:>20s}  mean={s['mean']:>12.4f}  std={s['std']:>12.4f}")

In [ ]:
import matplotlib.pyplot as plt

# Chose a variable and sample to plot
# See two cells above for a list of variables and sample count
# This is currently set to t2m for sample 5
input_var_choice = 2
output_var_choice = 0
sample_choice = 5

# Visualize one sample: input (low-res) vs output (high-res)
with xr.open_dataset(data_file, group="input") as ds_in:
    input_vars = list(ds_in.keys())
    sample_input = ds_in[input_vars[input_var_choice]][sample_choice].values

with xr.open_dataset(data_file, group="output") as ds_out:
    output_vars = list(ds_out.keys())
    sample_output = ds_out[output_vars[output_var_choice]][sample_choice].values

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

im0 = axes[0].imshow(sample_input, cmap="viridis", origin="lower")
axes[0].set_title(
    f"Input (low-res): {input_vars[input_var_choice]}\nShape: {sample_input.shape}"
)
plt.colorbar(im0, ax=axes[0], shrink=0.8)

im1 = axes[1].imshow(sample_output, cmap="viridis", origin="lower")
axes[1].set_title(
    f"Output (high-res): {output_vars[output_var_choice]}\nShape: {sample_output.shape}"
)
plt.colorbar(im1, ax=axes[1], shrink=0.8)

plt.suptitle(f"HRRR-Mini: Input vs Output (Sample {sample_choice})", fontsize=14)
plt.tight_layout()
plt.show()

print(f"\nUpsampling factor: {sample_output.shape[0] // sample_input.shape[0]}x")

---
## 4. Train the Regression Model

This section trains a **CorrDiff regression UNet** using `train.py` with Hydra overrides.

### Runtime-aware behavior

The notebook uses `compute_type` from Section 0:
- **GPU path (recommended)** trains the model on many images.
- **CPU path** uses a very short run so participants can still execute end-to-end.

Note: Results from either path are not expected to look physical. We train on only a couple thousand images; in a real setting, this would be millions.

### Parameter choices in this notebook

| Parameter | GPU Value | CPU Value | Why |
|-----------|-----------|-----------|-----|
| `batch_size_per_gpu` | 1 | 1 | Stable memory footprint |
| `total_batch_size` | 32 | 32 | Keeps optimizer behavior similar |
| `training_duration` | 2000 images | 2 images | Speed vs quality tradeoff (counts total images processed, not optimizer steps) |
| `fp_optimizations` | `amp-fp16` | `none` | GPU acceleration vs CPU compatibility |
| `save_checkpoint_freq` | 500 | 2 | Ensure a checkpoint exists quickly on CPU |
| `print_progress_freq` | 500 | 1 | More frequent logs for short CPU runs |
| `validation_freq` | 500 | 2 | Validation occurs during short CPU run |

### Architecture note (mini model)

- Base channels: 64
- Channel multipliers: [1, 2, 2]
- Attention resolution: 16
- Rough size: ~10M parameters (approximate; verify from model summary in your environment)

### Expected outputs

- Console training logs
- Checkpoints in `checkpoints_regression`
- At least one `.mdlus` checkpoint to use in inference

In [ ]:
CORRDIFF_DIR = "/content/physicsnemo/examples/weather/corrdiff"
import os

os.chdir(CORRDIFF_DIR)
print(f"Working directory: {os.getcwd()}")

In [ ]:
if compute_type == "gpu":
    print("Running GPU training configuration...")
    !python train.py \
        --config-name=config_training_hrrr_mini_regression.yaml \
        ++training.hp.batch_size_per_gpu=32 \
        ++training.hp.total_batch_size=32 \
        ++training.hp.training_duration=2000 \
        ++training.perf.fp_optimizations=amp-fp16 \
        ++training.io.save_checkpoint_freq=500 \
        ++training.io.print_progress_freq=500 \
        ++training.io.validation_freq=500 \
        ++wandb.mode=disabled
else:
    print("Running CPU fallback training configuration (shorter training run)...")
    !python train.py \
        --config-name=config_training_hrrr_mini_regression.yaml \
        ++training.hp.batch_size_per_gpu=1 \
        ++training.hp.total_batch_size=32 \
        ++training.hp.training_duration=2 \
        ++training.perf.fp_optimizations=none \
        ++training.io.save_checkpoint_freq=2 \
        ++training.io.print_progress_freq=1 \
        ++training.io.validation_freq=2 \
        ++wandb.mode=disabled

In [ ]:
# List saved checkpoints
import glob

CKPT_DIR = os.path.join(CORRDIFF_DIR, "checkpoints_regression")
# Sort by file modification time so the final entry is truly the latest checkpoint.
checkpoints = sorted(glob.glob(os.path.join(CKPT_DIR, "*.mdlus")), key=os.path.getmtime)

print(f"Checkpoint directory: {CKPT_DIR}")
print(f"Found {len(checkpoints)} checkpoint(s):")
for ckpt in checkpoints:
    size_mb = os.path.getsize(ckpt) / 1e6
    print(f"  {os.path.basename(ckpt)} ({size_mb:.1f} MB)")

LATEST_CKPT = checkpoints[-1] if checkpoints else None
print(f"\nLatest checkpoint: {LATEST_CKPT}")

---
## 5. Run Inference

Now we run `generate.py` to create a forecast in **regression-only** mode using the latest checkpoint from training.

### What this step does

- Loads your trained regression checkpoint
- Runs inference for a selected timestamp (`2020-01-01T02:00:00`)
- Writes outputs to NetCDF

### Output structure

| Group | Description |
|-------|-------------|
| `input` | Low-resolution fields presented to the model |
| `truth` | Ground-truth high-resolution targets from dataset |
| `prediction` | Model-generated high-resolution predictions |

### Success criteria

- No runtime error from `generate.py`
- At least one output `.nc` file produced
- `OUTPUT_NC` resolves to a valid path

In [ ]:
if LATEST_CKPT is None:
    raise RuntimeError("No checkpoint found. Run the training cell first.")

os.chdir(CORRDIFF_DIR)

!python generate.py \
    --config-name=config_generate_hrrr_mini.yaml \
    "++generation.io.reg_ckpt_filename={LATEST_CKPT}" \
    ++generation.inference_mode=regression \
    ++generation.num_ensembles=1 \
    ++generation.seed_batch_size=1 \
    "++generation.times=[2020-01-01T02:00:00]" \
    ++wandb.mode=disabled

In [ ]:
# Locate the output file
OUTPUT_DIR = os.path.join(CORRDIFF_DIR)
output_files = glob.glob(os.path.join(OUTPUT_DIR, "*.nc"))

if not output_files:
    # Hydra may nest the output one level deeper
    output_files = glob.glob(os.path.join(OUTPUT_DIR, "**", "*.nc"), recursive=True)

print(f"Output directory: {OUTPUT_DIR}")
for f in output_files:
    print(f"  {os.path.basename(f)} ({os.path.getsize(f)/1e6:.1f} MB)")

OUTPUT_NC = output_files[0] if output_files else None
print(f"\nUsing: {OUTPUT_NC}")

---
## 6. Visualize the Results

This section compares prediction vs truth channel-by-channel using maps and simple error metrics.

### What you will inspect

- Spatial maps for each output variable:
  - ground truth
  - model prediction
  - error field (`prediction - truth`)
- Scalar metrics per variable:
  - RMSE
  - MAE
  - Bias

### How to read the plots

- **Truth panel**: target pattern from dataset.
- **Prediction panel**: what the regression model produced.
- **Error panel**: where model over/under-predicts.

Larger coherent error structures usually indicate systematic bias; noisy local errors can indicate unresolved fine-scale details.

### How to use metrics

- **RMSE** emphasizes larger errors.
- **MAE** tracks average absolute error more robustly.
- **Bias** indicates directional drift (positive or negative).

### Notes

- Results are not expected to look physical given the minimal training, especially with CPU fallback options.
- For better quality, increase training duration and rerun training + inference. Feel free to experiment on your own when you have more time!

In [ ]:
import netCDF4 as nc
import matplotlib.pyplot as plt
import numpy as np

if OUTPUT_NC is None:
    raise RuntimeError("No output NetCDF found. Run inference first.")

ds = nc.Dataset(OUTPUT_NC, "r")

print("NetCDF groups:", list(ds.groups.keys()))
print("\nPrediction variables:")
for var in ds.groups["prediction"].variables:
    shape = ds.groups["prediction"][var].shape
    print(f"  {var}: {shape}")
print("\nTruth variables:")
for var in ds.groups["truth"].variables:
    shape = ds.groups["truth"][var].shape
    print(f"  {var}: {shape}")

In [ ]:
# Plot truth vs prediction for each output variable
truth_group = ds.groups["truth"]
pred_group = ds.groups["prediction"]

output_vars = [v for v in truth_group.variables if v not in ("lat", "lon", "time")]

fig, axes = plt.subplots(len(output_vars), 3, figsize=(16, 5 * len(output_vars)))
if len(output_vars) == 1:
    axes = axes[np.newaxis, :]  # ensure 2D indexing

for i, var in enumerate(output_vars):
    truth = truth_group[var][0]  # time=0
    pred = pred_group[var][0, 0]  # ensemble=0, time=0
    diff = pred - truth

    vmin = min(np.nanmin(truth), np.nanmin(pred))
    vmax = max(np.nanmax(truth), np.nanmax(pred))

    im0 = axes[i, 0].imshow(truth, cmap="RdBu_r", origin="lower", vmin=vmin, vmax=vmax)
    axes[i, 0].set_title(f"Truth: {var}")
    plt.colorbar(im0, ax=axes[i, 0], shrink=0.8)

    im1 = axes[i, 1].imshow(pred, cmap="RdBu_r", origin="lower", vmin=vmin, vmax=vmax)
    axes[i, 1].set_title(f"Prediction: {var}")
    plt.colorbar(im1, ax=axes[i, 1], shrink=0.8)

    dlim = max(abs(np.nanmin(diff)), abs(np.nanmax(diff)))
    im2 = axes[i, 2].imshow(diff, cmap="RdBu_r", origin="lower", vmin=-dlim, vmax=dlim)
    axes[i, 2].set_title(f"Error (Pred - Truth): {var}")
    plt.colorbar(im2, ax=axes[i, 2], shrink=0.8)

plt.suptitle("CorrDiff Regression: Truth vs Prediction", fontsize=16, y=1.01)
plt.tight_layout()
plt.show()

In [ ]:
# Compute quantitative error metrics
print("Error metrics per variable:")
print(f"{'Variable':>10s}  {'RMSE':>10s}  {'MAE':>10s}  {'Bias':>10s}")
print("-" * 46)

for var in output_vars:
    truth = np.array(truth_group[var][0])
    pred = np.array(pred_group[var][0, 0])
    diff = pred - truth

    rmse = np.sqrt(np.nanmean(diff**2))
    mae = np.nanmean(np.abs(diff))
    bias = np.nanmean(diff)
    print(f"{var:>10s}  {rmse:10.4f}  {mae:10.4f}  {bias:10.4f}")

ds.close()

---
## 7. Inspect the Input Fields

Before concluding, inspect the actual model inputs to build intuition about what information drives the prediction.

### Why this helps

Model output quality depends strongly on:
- dynamic meteorological channels (wind, temperature, humidity, pressure)
- static geographic context (latitude/longitude, elevation, land-sea mask)

Visualizing inputs can explain model behavior, especially when errors cluster by terrain or coastal regions. Perhaps more importantly, it can also help to debug if a model is not performing as expected (data could look incorrect).

### What this plot shows

- A panel per input variable at the selected timestep
- Relative spatial structure seen by the model

In [ ]:
ds = nc.Dataset(OUTPUT_NC, "r")
input_group = ds.groups["input"]
input_vars = [v for v in input_group.variables if v not in ("lat", "lon", "time")]

n_vars = len(input_vars)
cols = min(4, n_vars)
rows = (n_vars + cols - 1) // cols

fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
if n_vars == 1:
    axes = np.array([axes])
axes = np.atleast_2d(axes)

for i, var in enumerate(input_vars):
    r, c = divmod(i, cols)
    data = input_group[var][0]  # time=0
    im = axes[r, c].imshow(data, cmap="viridis", origin="lower")
    axes[r, c].set_title(var, fontsize=11)
    plt.colorbar(im, ax=axes[r, c], shrink=0.8)

# Hide unused axes
for i in range(n_vars, rows * cols):
    r, c = divmod(i, cols)
    axes[r, c].axis("off")

plt.suptitle("Input Fields Seen by the Model", fontsize=14)
plt.tight_layout()
plt.show()

ds.close()

---
## 8. Regional Transfer Run: Downscaling GFS Quarter-Degree Data

In this section, we take a fully trained regression model trained on HRRR-Mini earlier in the notebook and run it over a **different geographic region of the globe** using real-time GFS data as input. If you were to continue training your model for 2,000,000 samples, a similar skill would be achieved. 

### What this demonstrates

- The trained model can be applied to data from a new region without retraining.
- We can ingest coarse 0.25° GFS fields and produce finer-scale downscaled output over a user-selected region.
- This is a practical template for regional transfer experiments with CorrDiff.

### Domain setup

The domain is a **192 km x 192 km box** centered on a user-selected latitude/longitude, sized to match the spatial scale of the HRRR-Mini training data. The 64x64 high-resolution output grid has ~3 km cell spacing (matching HRRR native resolution), and the 8x8 low-resolution input grid has ~24 km cell spacing (close to GFS 0.25° native resolution of ~28 km). This keeps the spatial scales consistent with what the model learned during training.

### End-to-end workflow

These cells execute the following steps in order:

1. **Download and transform GFS data** — fetch the most recent GFS 0.25° analysis from NOAA NOMADS for the selected subregion, extract 26 meteorological input channels (surface fields + pressure-level fields at 1000/850/500/250 hPa), build 4 invariant fields (latitude, longitude, elevation, land-sea mask) from the same GFS data at ~3 km resolution, and pack everything into a grouped NetCDF matching the CorrDiff schema. This all ensures that the data matches what the model was trained on (variables included, resolution, etc.)
2. **Run regression inference** — execute `generate.py` with the trained checkpoint and the custom regional GFS file.
3. **Visualize predictions** — compare the coarse GFS input fields against the model's downscaled output fields.
4. **Inspect model inputs** — plot all 26 low-resolution input channels and the 4 invariant fields to verify the data fed to the model.

### Important scientific caveat

This is a **domain-shift experiment**. The checkpoint was trained on CONUS HRRR data, so running it on GFS data in a different region introduces changes in geography, data source, and normalization statistics. The results are useful for demonstrating the workflow and inspecting qualitative behavior, but should not be treated as calibrated operational output. For stronger regional skill, it is common to retrain or fine-tune with region-consistent data.

### 8.1 Download GFS data and build the CorrDiff input file

The next 3 cells handle the entire data loading pipeline:

1. **Set the center location**: At the top of the cell, choose `REGION_CENTER_LAT` and `REGION_CENTER_LON`. These values define the center of the 192 km x 192 km domain. Defaults are set to 45.81, 15.98 (Zagreb, Croatia)

2. **Download**: Fetches the most recent GFS 0.25° analysis from [NOAA NOMADS](https://nomads.ncep.noaa.gov/) as a GRIB2 file, filtered to the selected subregion. It tries recent cycles (18z, 12z, 06z, 00z) going back up to 3 days until it finds an available file.

3. **Extract input channels** (8x8 low-res, ~24 km per cell): Reads 26 meteorological fields from the GRIB — 6 surface variables (10m winds, 2m temperature, total column water vapor, surface pressure, mean sea-level pressure) and 20 pressure-level variables (U/V wind, temperature, specific humidity, geopotential at 1000/850/500/250 hPa). Each field is bilinearly interpolated from the native 0.25° grid onto the 8x8 coarse input tile.

4. **Build invariant fields** (64x64 high-res, ~3 km per cell): Constructs latitude, longitude, surface elevation, and land-sea mask grids at the 64x64 output resolution over the selected domain, derived from the same GFS data.

5. **Assemble grouped NetCDF**: Packs the input, output (placeholder from HRRR-Mini for schema compatibility), invariant, and root groups into a single NetCDF file that matches the structure expected by the CorrDiff dataset loader.

6. **NaN check**: Validates that no fields contain missing values, which would propagate through the model and produce empty outputs.

In [ ]:
import os
import urllib.parse
from datetime import datetime, timedelta, timezone

import numpy as np
import requests
import xarray as xr
import cfgrib


DATA_DIR = os.path.join(CORRDIFF_DIR, "data", "hrrr_mini")
SOURCE_FILE = os.path.join(DATA_DIR, "hrrr_mini_train.nc")
CUSTOM_FILE = os.path.join(DATA_DIR, "hrrr_mini_train_gfs_region.nc")
OUTPUT_NC_GFS = os.path.join(DATA_DIR, "gfs_region_output.nc")

# User-configurable center point for regional transfer runs (Bangkok, Thailand).
REGION_CENTER_LAT = 13.75
REGION_CENTER_LON = 100.50

# Match HRRR-Mini training scale: 64x64 at ~3 km -> 192 km total
TARGET_RES_KM = 3.0
HR_GRID = 64
DOMAIN_KM = TARGET_RES_KM * HR_GRID

KM_PER_DEG_LAT = 111.0
KM_PER_DEG_LON = 111.0 * np.cos(np.radians(REGION_CENTER_LAT))
LAT_SPAN_DEG = DOMAIN_KM / KM_PER_DEG_LAT
LON_SPAN_DEG = DOMAIN_KM / KM_PER_DEG_LON

REGION_BBOX = {
    "leftlon": REGION_CENTER_LON - (LON_SPAN_DEG / 2.0),
    "rightlon": REGION_CENTER_LON + (LON_SPAN_DEG / 2.0),
    "toplat": REGION_CENTER_LAT + (LAT_SPAN_DEG / 2.0),
    "bottomlat": REGION_CENTER_LAT - (LAT_SPAN_DEG / 2.0),
}

PAD = 0.5
DOWNLOAD_BBOX = {
    "leftlon": REGION_BBOX["leftlon"] - PAD,
    "rightlon": REGION_BBOX["rightlon"] + PAD,
    "toplat": REGION_BBOX["toplat"] + PAD,
    "bottomlat": REGION_BBOX["bottomlat"] - PAD,
}

INPUT_VARS = [
    "u10m",
    "v10m",
    "t2m",
    "tcwv",
    "sp",
    "msl",
    "u1000",
    "u850",
    "u500",
    "u250",
    "v1000",
    "v850",
    "v500",
    "v250",
    "z1000",
    "z850",
    "z500",
    "z250",
    "t1000",
    "t850",
    "t500",
    "t250",
    "q1000",
    "q850",
    "q500",
    "q250",
]

print(
    f"Domain: {DOMAIN_KM:.0f} km x {DOMAIN_KM:.0f} km  ({LAT_SPAN_DEG:.2f} deg lat x {LON_SPAN_DEG:.2f} deg lon)"
)
print(
    f"  64x64 output res: ~{TARGET_RES_KM:.0f} km    8x8 input res: ~{TARGET_RES_KM * 8:.0f} km"
)
print(f"  Center: {REGION_CENTER_LAT:.2f}N, {REGION_CENTER_LON:.2f}E")
print(
    f"  Bbox: lat [{REGION_BBOX['bottomlat']:.2f}, {REGION_BBOX['toplat']:.2f}], "
    f"lon [{REGION_BBOX['leftlon']:.2f}, {REGION_BBOX['rightlon']:.2f}]"
)


MAJOR_CITIES = {
    # Southeast Asia
    "Bangkok": (13.7563, 100.5018),
    "Chiang Mai": (18.7883, 98.9853),
    "Phuket": (7.8804, 98.3923),
    "Pattaya": (12.9236, 100.8825),
    "Kuala Lumpur": (3.1390, 101.6869),
    "Singapore": (1.3521, 103.8198),
    "Yangon": (16.8661, 96.1951),
    "Ho Chi Minh City": (10.8231, 106.6297),
    "Hanoi": (21.0285, 105.8542),
    "Phnom Penh": (11.5564, 104.9282),
    "Vientiane": (17.9757, 102.6331),
}


def coarse_lat_lon_grid(n=8):
    """Build 2D lat/lon cell-center grids for the coarse input tile."""
    lat_1d = np.linspace(REGION_BBOX["toplat"], REGION_BBOX["bottomlat"], n)
    lon_1d = np.linspace(REGION_BBOX["leftlon"], REGION_BBOX["rightlon"], n)
    lon_2d, lat_2d = np.meshgrid(lon_1d, lat_1d)
    return lat_2d.astype(np.float32), lon_2d.astype(np.float32)


def cities_in_bbox(pad_deg=0.35):
    """Return major cities near the selected regional domain."""
    lat_min = REGION_BBOX["bottomlat"] - pad_deg
    lat_max = REGION_BBOX["toplat"] + pad_deg
    lon_min = REGION_BBOX["leftlon"] - pad_deg
    lon_max = REGION_BBOX["rightlon"] + pad_deg
    return {
        name: (lat, lon)
        for name, (lat, lon) in MAJOR_CITIES.items()
        if lat_min <= lat <= lat_max and lon_min <= lon <= lon_max
    }

In [ ]:
def build_nomads_filter_url(
    date_yyyymmdd: str, cycle_hh: str, endpoint: str, file_name: str
) -> str:
    base = f"https://nomads.ncep.noaa.gov/cgi-bin/{endpoint}"
    params = {
        "file": file_name,
        "lev_10_m_above_ground": "on",
        "lev_2_m_above_ground": "on",
        "lev_surface": "on",
        "lev_mean_sea_level": "on",
        "lev_entire_atmosphere_(considered_as_a_single_layer)": "on",
        "lev_1000_mb": "on",
        "lev_850_mb": "on",
        "lev_500_mb": "on",
        "lev_250_mb": "on",
        "var_UGRD": "on",
        "var_VGRD": "on",
        "var_TMP": "on",
        "var_SPFH": "on",
        "var_HGT": "on",
        "var_PRES": "on",
        "var_PRMSL": "on",
        "var_PWAT": "on",
        "var_LAND": "on",
        "subregion": "",
        "leftlon": str(DOWNLOAD_BBOX["leftlon"]),
        "rightlon": str(DOWNLOAD_BBOX["rightlon"]),
        "toplat": str(DOWNLOAD_BBOX["toplat"]),
        "bottomlat": str(DOWNLOAD_BBOX["bottomlat"]),
        "dir": f"/gfs.{date_yyyymmdd}/{cycle_hh}/atmos",
    }
    return f"{base}?{urllib.parse.urlencode(params)}"


def _download_grib(url, out_path, min_size_mb=0.1):
    tmp = f"{out_path}.tmp"
    try:
        r = requests.get(
            url, stream=True, timeout=300, headers={"User-Agent": "Mozilla/5.0"}
        )
        if r.status_code != 200:
            print(f"  HTTP {r.status_code}")
            return False
        with open(tmp, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)
        size_mb = os.path.getsize(tmp) / 1e6
        with open(tmp, "rb") as f:
            magic = f.read(4)
        if magic != b"GRIB":
            print(f"  Not a valid GRIB file")
            os.remove(tmp)
            return False
        if size_mb < min_size_mb:
            print(f"  File too small ({size_mb:.3f} MB < {min_size_mb} MB)")
            os.remove(tmp)
            return False
        os.replace(tmp, out_path)
        print(f"  OK ({size_mb:.1f} MB)")
        return True
    except Exception as e:
        print(f"  Error: {e}")
        if os.path.exists(tmp):
            os.remove(tmp)
        return False


def download_recent_gfs(out_path, max_days_back=3):
    os.makedirs(os.path.dirname(out_path), exist_ok=True)
    now = datetime.now(timezone.utc)
    for day_offset in range(max_days_back + 1):
        date_str = (now - timedelta(days=day_offset)).strftime("%Y%m%d")
        for cycle in ["18", "12", "06", "00"]:
            fname = f"gfs.t{cycle}z.pgrb2.0p25.f000"
            for endpoint in ["filter_gfs_0p25_1hr.pl", "filter_gfs_0p25.pl"]:
                print(f"{date_str} {cycle}z  filter:{endpoint}")
                url = build_nomads_filter_url(date_str, cycle, endpoint, fname)
                if _download_grib(url, out_path, min_size_mb=0.001):
                    return out_path
            direct = f"https://nomads.ncep.noaa.gov/pub/data/nccf/com/gfs/prod/gfs.{date_str}/{cycle}/atmos/{fname}"
            print(f"{date_str} {cycle}z  direct:nomads.ncep.noaa.gov")
            if _download_grib(direct, out_path, min_size_mb=5.0):
                return out_path
    raise RuntimeError("Could not download GFS data. Try rerunning later.")


def get_first_var(datasets, names, required_dim=None, exclude_dim=None):
    for name in names:
        for ds in datasets:
            if name in ds.data_vars:
                if required_dim is not None and required_dim not in ds.dims:
                    continue
                if exclude_dim is not None and exclude_dim in ds.dims:
                    continue
                return ds[name]
    raise KeyError(f"Could not find {names}")


def interpolate_to_grid(da, lat_target, lon_target):
    d = da
    for dim in [
        "time",
        "step",
        "heightAboveGround",
        "surface",
        "meanSea",
        "entireAtmosphere",
        "valid_time",
    ]:
        if dim in d.dims and d.sizes[dim] == 1:
            d = d.isel({dim: 0})

    lat_name = "latitude" if "latitude" in d.coords else "lat"
    lon_name = "longitude" if "longitude" in d.coords else "lon"

    lon_vals = (((d[lon_name] + 180) % 360) - 180).values
    d = d.assign_coords({lon_name: lon_vals}).sortby(lon_name)
    d = d.sortby(lat_name, ascending=False)

    patch = d.interp(
        {lat_name: lat_target, lon_name: lon_target},
        method="linear",
        kwargs={"fill_value": "extrapolate"},
    )
    return np.asarray(patch.values, dtype=np.float32)


def extract_valid_time(groups):
    for ds in groups:
        for coord in ("valid_time", "time"):
            if coord in ds.coords:
                t = ds.coords[coord].values
                return np.datetime64(t.flat[0] if isinstance(t, np.ndarray) else t, "s")
    return np.datetime64("now", "s")

In [ ]:
import warnings

warnings.filterwarnings("ignore", category=FutureWarning, module="cfgrib")

grib_file = os.path.join(DATA_DIR, "gfs_region_subset.grib2")
grib_file = download_recent_gfs(grib_file)

all_groups = cfgrib.open_datasets(grib_file, backend_kwargs={"indexpath": ""})
print(f"Loaded {len(all_groups)} cfgrib group(s)")

# ── Extract and interpolate all 26 input channels onto the 8x8 coarse grid ──

SURFACE_FIELDS = {
    "u10m": ["u10", "u10m"],
    "v10m": ["v10", "v10m"],
    "t2m": ["t2m", "2t"],
    "tcwv": ["pwat", "tcwv"],
    "sp": ["sp", "pres"],
    "msl": ["msl", "prmsl"],
}

PL_FIELDS = {
    "u": get_first_var(all_groups, ["u"], required_dim="isobaricInhPa"),
    "v": get_first_var(all_groups, ["v"], required_dim="isobaricInhPa"),
    "t": get_first_var(all_groups, ["t"], required_dim="isobaricInhPa"),
    "q": get_first_var(all_groups, ["q"], required_dim="isobaricInhPa"),
}
gh_pl = get_first_var(all_groups, ["gh", "z"], required_dim="isobaricInhPa")

lat_target = np.linspace(REGION_BBOX["toplat"], REGION_BBOX["bottomlat"], 8)
lon_target = np.linspace(REGION_BBOX["leftlon"], REGION_BBOX["rightlon"], 8)

print(
    f"8x8 tile midpoint: lat={(lat_target[3]+lat_target[4])/2:.2f}, "
    f"lon={(lon_target[3]+lon_target[4])/2:.2f}"
)

arr = {}
for channel, aliases in SURFACE_FIELDS.items():
    arr[channel] = interpolate_to_grid(
        get_first_var(all_groups, aliases), lat_target, lon_target
    )

LEVELS = [1000, 850, 500, 250]
for lev in LEVELS:
    for prefix, da in PL_FIELDS.items():
        arr[f"{prefix}{lev}"] = interpolate_to_grid(
            da.sel(isobaricInhPa=lev), lat_target, lon_target
        )
    # geopotential (m²/s²) = geopotential height (m) × g
    arr[f"z{lev}"] = interpolate_to_grid(
        gh_pl.sel(isobaricInhPa=lev) * 9.80665, lat_target, lon_target
    )

custom_time = extract_valid_time(all_groups)
print(f"Using GFS valid time: {custom_time}")

custom_time_str = str(custom_time)

# ── Assemble grouped NetCDF ──

input_ds = xr.Dataset(
    {var: (("time", "y", "x"), arr[var][None, :, :]) for var in INPUT_VARS},
    coords={"time": [custom_time], "y": np.arange(8), "x": np.arange(8)},
)

with xr.open_dataset(SOURCE_FILE, group="output") as ds_out:
    first_dim = list(ds_out.dims)[0]
    output_ds = ds_out[["2t", "10u", "10v", "tp"]].isel({first_dim: slice(0, 1)}).load()
if first_dim != "time":
    output_ds = output_ds.rename({first_dim: "time"})
output_ds = output_ds.assign_coords(time=[custom_time])

lat_hr = np.linspace(REGION_BBOX["toplat"], REGION_BBOX["bottomlat"], 64)
lon_hr = np.linspace(REGION_BBOX["leftlon"], REGION_BBOX["rightlon"], 64)
lon_grid, lat_grid = np.meshgrid(lon_hr, lat_hr)

hgt_sfc = get_first_var(all_groups, ["gh", "orog", "z"], exclude_dim="isobaricInhPa")
lsm_gfs = get_first_var(all_groups, ["lsm", "land"])

invariant_ds = xr.Dataset(
    {
        "latitude": (("y", "x"), lat_grid.astype(np.float32)),
        "longitude": (("y", "x"), lon_grid.astype(np.float32)),
        "elev_mean": (("y", "x"), interpolate_to_grid(hgt_sfc, lat_hr, lon_hr)),
        "lsm_mean": (("y", "x"), interpolate_to_grid(lsm_gfs, lat_hr, lon_hr)),
    }
)

root_ds = xr.Dataset(
    {"coord": (("time", "ij"), np.array([[0, 0]], dtype=np.int64))},
    coords={"time": [custom_time]},
)
root_ds.to_netcdf(CUSTOM_FILE, mode="w")
input_ds.to_netcdf(CUSTOM_FILE, mode="a", group="input")
output_ds.to_netcdf(CUSTOM_FILE, mode="a", group="output")
invariant_ds.to_netcdf(CUSTOM_FILE, mode="a", group="invariant")

print(f"Wrote: {CUSTOM_FILE}")

# ── NaN sanity check ──
for grp_name in ["input", "output", "invariant"]:
    with xr.open_dataset(CUSTOM_FILE, group=grp_name) as ds_check:
        for v in ds_check.data_vars:
            n_nan = int(np.isnan(ds_check[v].values).sum())
            if n_nan > 0:
                print(f"  WARNING: {grp_name}/{v} has {n_nan} NaN values!")
print("NaN check complete.")

### 8.2 Run regression inference on the selected region

This cell runs `generate.py` with the trained regression checkpoint, pointing it at the custom regional GFS NetCDF file built above. The Hydra overrides direct the model to:

- Use regression-only inference (no diffusion stage).
- Process the single GFS timestep we downloaded.
- Write predictions to a NetCDF output file.

Because this is a domain-shift experiment, the model applies HRRR-trained normalization statistics to GFS data. The spatial patterns in the output should still be physically plausible, but absolute values may be offset.

In [ ]:
import glob
import urllib.request

CKPT_FILENAME = "CorrDiffRegressionUNet.0.1300000.mdlus"
CKPT_DIR = os.path.join(os.path.dirname(CORRDIFF_DIR), "ckpts")
PRETRAINED_CKPT = os.path.join(CKPT_DIR, CKPT_FILENAME)

if not os.path.exists(PRETRAINED_CKPT):
    os.makedirs(CKPT_DIR, exist_ok=True)
    ckpt_url = f"https://github.com/maxfield-green/ai_hackathons/raw/main/ckpts/{CKPT_FILENAME}"
    print(f"Downloading pretrained checkpoint from GitHub (~42 MB)...")
    urllib.request.urlretrieve(ckpt_url, PRETRAINED_CKPT)
    print(f"Saved to {PRETRAINED_CKPT} ({os.path.getsize(PRETRAINED_CKPT) / 1e6:.1f} MB)")
else:
    print(f"Checkpoint already exists: {PRETRAINED_CKPT}")

if not os.path.exists(CUSTOM_FILE):
    raise RuntimeError("Custom regional GFS file not found. Run the previous cell first.")

os.chdir(CORRDIFF_DIR)

!python generate.py \
    --config-name=config_generate_hrrr_mini.yaml \
    "++generation.io.reg_ckpt_filename={PRETRAINED_CKPT}" \
    "++dataset.data_path={CUSTOM_FILE}" \
    ++generation.inference_mode=regression \
    ++generation.num_ensembles=1 \
    ++generation.seed_batch_size=1 \
    "++generation.times=[{custom_time_str}]" \
    ++wandb.mode=disabled

# Pick latest NetCDF produced by generate.py (exclude data/ inputs)
data_dir_prefix = os.path.join(CORRDIFF_DIR, "data")
nc_candidates = sorted(
    [f for f in glob.glob(os.path.join(CORRDIFF_DIR, "**", "*.nc"), recursive=True)
     if not f.startswith(data_dir_prefix)],
    key=os.path.getmtime,
)

OUTPUT_NC_GFS = nc_candidates[-1] if nc_candidates else None
print(f"Regional GFS inference output: {OUTPUT_NC_GFS}")

### 8.3 Visualize GFS input vs downscaled predictions

This plot compares the native coarse GFS input, the upsampled model input, and the CorrDiff prediction for 10m wind components (`10u`, `10v`).

- **Left column**: Native GFS input on the untouched 8x8 tile (~24 km spacing) from `CUSTOM_FILE`.
- **Middle column**: The same case after CorrDiff's bilinear upsampling to 64x64, which is what the regression model actually receives.
- **Right column**: CorrDiff regression prediction on the 64x64 output grid (~3 km spacing).

All three columns are aligned to the same regional GFS timestep and plotted on lat/lon with Cartopy coastlines, borders, and nearby major cities. Re-run the inference cell above if the prediction file is missing or stale.

Note: Due to the short training period and coarse invariant features, we do not expect to see much difference for this demonstration. Further training and finer invariant fields should greatly improve these results.

In [ ]:
import os
import sys

import cartopy.crs as ccrs
import cartopy.feature as cfeature
import netCDF4 as nc
import matplotlib.pyplot as plt
import numpy as np
import xarray as xr

if OUTPUT_NC_GFS is None or not os.path.exists(OUTPUT_NC_GFS):
    raise RuntimeError(
        "Regional GFS inference output not found. Run the inference cell above first."
    )

if not os.path.exists(CUSTOM_FILE):
    raise RuntimeError(
        "Custom GFS input file not found. Run the regional GFS prep cell first."
    )

if CORRDIFF_DIR not in sys.path:
    sys.path.insert(0, CORRDIFF_DIR)
from datasets.hrrrmini import _zoom_extrapolate


def upsample_native(field_8x8):
    """Match HRRRMiniDataset upsampling: 8x8 -> 64x64 bilinear."""
    x = field_8x8[np.newaxis, ...].astype(np.float32)
    y = np.empty((1, 64, 64), dtype=np.float32)
    _zoom_extrapolate(x, y, 8)
    return y[0]


def plot_map_panel(
    ax,
    field,
    lat_2d,
    lon_2d,
    title,
    cmap="RdBu_r",
    vmin=None,
    vmax=None,
    add_cities=False,
):
    """Plot a field on lat/lon with coastlines, borders, and optional city markers."""
    mesh = ax.pcolormesh(
        lon_2d,
        lat_2d,
        field,
        transform=ccrs.PlateCarree(),
        cmap=cmap,
        vmin=vmin,
        vmax=vmax,
        shading="auto",
    )

    lon_min, lon_max = float(lon_2d.min()), float(lon_2d.max())
    lat_min, lat_max = float(lat_2d.min()), float(lat_2d.max())
    pad_lon = max(0.05, (lon_max - lon_min) * 0.06)
    pad_lat = max(0.05, (lat_max - lat_min) * 0.06)
    ax.set_extent(
        [lon_min - pad_lon, lon_max + pad_lon, lat_min - pad_lat, lat_max + pad_lat],
        crs=ccrs.PlateCarree(),
    )
    ax.add_feature(cfeature.COASTLINE.with_scale("10m"), linewidth=0.7)
    ax.add_feature(cfeature.BORDERS.with_scale("10m"), linewidth=0.5, linestyle=":")
    gl = ax.gridlines(draw_labels=True, linewidth=0.3, alpha=0.45, linestyle="--")
    gl.top_labels = False
    gl.right_labels = False
    ax.set_title(title)

    if add_cities:
        for name, (city_lat, city_lon) in cities_in_bbox().items():
            ax.plot(
                city_lon,
                city_lat,
                marker="o",
                color="black",
                markersize=4,
                transform=ccrs.PlateCarree(),
                zorder=5,
            )
            ax.text(
                city_lon + 0.03,
                city_lat,
                name,
                transform=ccrs.PlateCarree(),
                fontsize=8,
                ha="left",
                va="center",
                bbox=dict(boxstyle="round,pad=0.15", fc="white", alpha=0.75, lw=0),
                zorder=6,
            )

    return mesh


plot_vars = ["10u", "10v"]
input_var_map = {"10u": "u10m", "10v": "v10m"}

with xr.open_dataset(CUSTOM_FILE, group="input") as ds_native:
    native_time = np.datetime64(ds_native["time"].values[0], "s")
    native_inputs = {
        var: ds_native[input_var_map[var]].isel(time=0).values.astype(np.float32)
        for var in plot_vars
    }

with xr.open_dataset(CUSTOM_FILE, group="invariant") as ds_inv:
    lat_hr = ds_inv["latitude"].values
    lon_hr = ds_inv["longitude"].values

lat_lr, lon_lr = coarse_lat_lon_grid(8)

ds_gfs = nc.Dataset(OUTPUT_NC_GFS, "r")
pred_group = ds_gfs.groups["prediction"]

fig, axes = plt.subplots(
    len(plot_vars),
    3,
    figsize=(18, 5.5 * len(plot_vars)),
    subplot_kw={"projection": ccrs.PlateCarree()},
    constrained_layout=True,
)
if len(plot_vars) == 1:
    axes = np.array([axes])

for i, var in enumerate(plot_vars):
    native = native_inputs[var]
    model_input = upsample_native(native)
    pred = np.array(pred_group[var][0, 0])

    vmin = min(
        float(np.nanmin(native)),
        float(np.nanmin(model_input)),
        float(np.nanmin(pred)),
    )
    vmax = max(
        float(np.nanmax(native)),
        float(np.nanmax(model_input)),
        float(np.nanmax(pred)),
    )

    # Always pass add_cities=True to all panels to show city points on all 3 subplots
    panels = [
        (
            native,
            lat_lr,
            lon_lr,
            f"Native GFS Input (8x8, ~24 km): {input_var_map[var]}",
            True,
        ),
        (
            model_input,
            lat_hr,
            lon_hr,
            f"Model Input (64x64 upsampled): {input_var_map[var]}",
            True,
        ),
        (pred, lat_hr, lon_hr, f"CorrDiff Prediction (64x64, ~3 km): {var}", True),
    ]

    for j, (field, lat_2d, lon_2d, title, add_cities) in enumerate(panels):
        mesh = plot_map_panel(
            axes[i, j],
            field,
            lat_2d,
            lon_2d,
            title,
            vmin=vmin,
            vmax=vmax,
            add_cities=add_cities,
        )
        plt.colorbar(mesh, ax=axes[i, j], shrink=0.85, pad=0.02)

fig.suptitle(
    f"Regional Downscaling at {native_time}: Native Input -> Model Input -> Prediction",
    fontsize=15,
)
plt.show()

ds_gfs.close()

### 8.4 Inspect model inputs and invariant fields

This cell plots every channel that was fed to the model, organized into two groups:

- **Input channels (8x8)**: The 26 low-resolution meteorological fields extracted from GFS — surface variables in the first row, then pressure-level U-wind, V-wind, geopotential, temperature, and specific humidity across the remaining rows. These are what the model "sees" as its coarse input.

- **Invariant fields (64x64)**: The four static fields at output resolution (~3 km per cell) — latitude, longitude, elevation (from GFS surface geopotential height), and land-sea mask (from GFS LAND field). These cover the 192 km x 192 km selected domain and provide the model with geographic context for downscaling.

In [ ]:
import xarray as xr
import matplotlib.pyplot as plt
import numpy as np

# ── Load the custom regional GFS file ──
input_ds = xr.open_dataset(CUSTOM_FILE, group="input")
invar_ds = xr.open_dataset(CUSTOM_FILE, group="invariant")

# Logical groupings for the 26 low-res input channels
input_groups = {
    "Surface": ["u10m", "v10m", "t2m", "tcwv", "sp", "msl"],
    "U-wind (hPa)": ["u1000", "u850", "u500", "u250"],
    "V-wind (hPa)": ["v1000", "v850", "v500", "v250"],
    "Geopotential (hPa)": ["z1000", "z850", "z500", "z250"],
    "Temperature (hPa)": ["t1000", "t850", "t500", "t250"],
    "Specific humidity": ["q1000", "q850", "q500", "q250"],
}

flat_vars = [v for grp in input_groups.values() for v in grp]

ncols = 6
nrows = int(np.ceil(len(flat_vars) / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols * 3, nrows * 2.8))
axes = axes.flatten()

for idx, var in enumerate(flat_vars):
    ax = axes[idx]
    data = input_ds[var].isel(time=0).values
    im = ax.imshow(data, origin="upper", cmap="viridis")
    ax.set_title(var, fontsize=10)
    ax.set_xticks([])
    ax.set_yticks([])
    fig.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

for idx in range(len(flat_vars), len(axes)):
    axes[idx].axis("off")

fig.suptitle("Model Inputs — Regional GFS (8×8, ~24 km)", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

# ── Invariant fields (64×64 high-res) ──
invar_names = ["latitude", "longitude", "elev_mean", "lsm_mean"]
cmaps = ["coolwarm", "coolwarm", "terrain", "BrBG"]

fig2, axes2 = plt.subplots(1, 4, figsize=(16, 3.5))
for ax, name, cmap in zip(axes2, invar_names, cmaps):
    data = invar_ds[name].values
    im = ax.imshow(data, origin="upper", cmap=cmap)
    ax.set_title(name, fontsize=11)
    ax.set_xticks([])
    ax.set_yticks([])
    fig2.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

fig2.suptitle("Invariant Fields — Regional GFS (64×64, ~3 km)", fontsize=14, y=1.02)
plt.tight_layout()
plt.show()

input_ds.close()
invar_ds.close()

---
## 9. Summary and Next Steps

### What you completed in this notebook

1. Connected to runtime hardware (GPU/CPU).
2. Installed PhysicsNeMo + CorrDiff dependencies in Google Colab.
3. Downloaded and inspected "HRRR-Mini" grouped NetCDF data.
4. Trained a CorrDiff regression UNet deterministic ML model.
5. Generated regression-only predictions from a trained checkpoint.
6. Visualized fields and computed quick quantitative error metrics.
7. Inspected input channels to contextualize model behavior.
8. Downloaded real-time GFS 0.25° data from NOAA and constructed a CorrDiff-compatible NetCDF for a 192 km user-selected domain at ~3 km output resolution.
9. Ran the trained regression model on the regional GFS input as a domain-shift transfer experiment.
10. Visualized the downscaled predictions and inspected all 26 input channels and 4 invariant fields.

### What the regional transfer section showed

- A model trained on CONUS HRRR data can be executed on coarse resolution data in other regions.
- This gives a concrete, reproducible pattern for adapting the workflow to new regions and custom data pipelines.

### Practical interpretation

- We trained a model for a very short period of time. Results were not expected to score well. This was meant as a demonstration of how to run this pipeline end to end.
- The regional transfer run demonstrates portability of the pipeline, while also highlighting domain-shift caveats.

### Recommended next experiments

- **Train longer**: Increase `training_duration` substantially for stronger skill.
- **Tune model size**: Move from `mini` to `normal` when VRAM allows.
- **Add diffusion stage**: Extend from deterministic regression to probabilistic generation.
- **Evaluate systematically**: Compute regional validation metrics over multiple timestamps and events.

### Knobs to tune

| Parameter | Where | Practical effect |
|-----------|-------|------------------|
| `training.hp.training_duration` | `train.py` overrides | More updates, usually better quality, longer runtime |
| `training.hp.lr` | `train.py` overrides | Convergence speed/stability trade-off |
| `model_size` | config | Capacity vs memory/runtime |
| `generation.num_ensembles` | `generate.py` overrides | Number of generated members |
| `generation.times` | `generate.py` overrides | Which timestamps are evaluated |

### References

- [CorrDiff paper (arXiv 2309.15214)](https://arxiv.org/abs/2309.15214)
- [PhysicsNeMo GitHub](https://github.com/NVIDIA/physicsnemo)